# OBB_HA_HB - YOLO26m OBB Training from project.zip

Bu notebook sizin mevcut Drive yapiniza gore `project.zip` dosyasini kullanir.

Beklenen Drive yapisi:

```text
Drive'im/AOD4/
├── AOD4_GitHub/
│   └── OBB_HA_HB/
├── project.zip
└── OBB_HA_HB_colab.ipynb
```

Not: OBB egitimi icin `project.zip` icindeki label dosyalari OBB formatinda olmali: `class x1 y1 x2 y2 x3 y3 x4 y4`. Notebook bunu egitimden once kontrol eder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

AOD4_ROOT = Path('/content/drive/MyDrive/AOD4')
REPO_ROOT = AOD4_ROOT / 'AOD4_GitHub'
OBB_ROOT = REPO_ROOT / 'OBB_HA_HB'
ZIP_PATH = AOD4_ROOT / 'project.zip'
WORK_ROOT = Path('/content/aod4_obb_work')
PROJECT_ROOT = WORK_ROOT / 'project'

paths = {
    'AOD4 root': AOD4_ROOT,
    'repo': REPO_ROOT,
    'OBB flow': OBB_ROOT,
    'project.zip': ZIP_PATH,
    'OBB train script': OBB_ROOT / 'scripts' / 'train_yolo.py',
}

for label, path in paths.items():
    print(f"{label:18}:", 'OK' if path.exists() else 'MISS', path)

missing = [f'{label}: {path}' for label, path in paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Eksik dosya/klasor var:\n' + '\n'.join(missing))

## project.zip Ac

In [ ]:
!rm -rf /content/aod4_obb_work
!mkdir -p /content/aod4_obb_work
!unzip -q "/content/drive/MyDrive/AOD4/project.zip" -d /content/aod4_obb_work
!find /content/aod4_obb_work -maxdepth 2 -type d | sort | head -n 40

In [ ]:
from pathlib import Path

if not PROJECT_ROOT.exists():
    candidates = [p for p in WORK_ROOT.iterdir() if p.is_dir()]
    if len(candidates) == 1:
        PROJECT_ROOT = candidates[0]
    else:
        raise FileNotFoundError(f'Zip icinde project klasoru bulunamadi: {candidates}')

DATASET_ROOT = PROJECT_ROOT / 'dataset'
DATA_YAML = PROJECT_ROOT / 'configs' / 'data.yaml'

required = [
    DATASET_ROOT / 'images' / 'train',
    DATASET_ROOT / 'images' / 'val',
    DATASET_ROOT / 'images' / 'test',
    DATASET_ROOT / 'labels' / 'train',
    DATASET_ROOT / 'labels' / 'val',
    DATASET_ROOT / 'labels' / 'test',
    DATA_YAML,
]

for path in required:
    print('OK  ' if path.exists() else 'MISS', path)

missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('project.zip icindeki dataset yapisi eksik:\n' + '\n'.join(map(str, missing)))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_YAML:', DATA_YAML)

## OBB Label Hazirlama

`project.zip` icindeki label'lar 5 alanli normal YOLO bbox veya cok noktalı polygon formatindaysa notebook bunlari `project_obb/` altinda OBB corner formatina cevirir. Bu donusum axis-aligned OBB uretir: `class x1 y1 x2 y2 x3 y3 x4 y4`.

In [ ]:
import shutil

OBB_PROJECT_ROOT = WORK_ROOT / 'project_obb'
OBB_DATASET_ROOT = OBB_PROJECT_ROOT / 'dataset'
OBB_DATA_YAML = OBB_PROJECT_ROOT / 'configs' / 'data.yaml'

def find_first_non_empty_label(labels_dir: Path) -> Path:
    for label_path in sorted(labels_dir.glob('*.txt')):
        if label_path.read_text(encoding='utf-8').strip():
            return label_path
    raise FileNotFoundError(f'Bos olmayan label bulunamadi: {labels_dir}')

def label_line_to_obb_line(line: str) -> str:
    parts = line.strip().split()
    if not parts:
        return ''
    if len(parts) == 9:
        return line.strip()
    cls = parts[0]
    values = [float(value) for value in parts[1:]]
    if len(parts) == 5:
        x, y, w, h = values
        min_x = x - w / 2
        min_y = y - h / 2
        max_x = x + w / 2
        max_y = y + h / 2
    elif len(values) >= 6 and len(values) % 2 == 0:
        xs = values[0::2]
        ys = values[1::2]
        min_x = min(xs)
        min_y = min(ys)
        max_x = max(xs)
        max_y = max(ys)
    else:
        raise ValueError(f'Beklenmeyen label satiri: {line}')
    x1 = max(0.0, min_x)
    y1 = max(0.0, min_y)
    x2 = min(1.0, max_x)
    y2 = y1
    x3 = min(1.0, max_x)
    y3 = min(1.0, max_y)
    x4 = x1
    y4 = y3
    coords = [x1, y1, x2, y2, x3, y3, x4, y4]
    return cls + ' ' + ' '.join(f'{value:.10f}' for value in coords)

sample_label = find_first_non_empty_label(DATASET_ROOT / 'labels' / 'train')
sample_line = sample_label.read_text(encoding='utf-8').strip().splitlines()[0]
field_count = len(sample_line.split())

print('source sample label:', sample_label)
print('source sample line:', sample_line)
print('source field count:', field_count)

if OBB_PROJECT_ROOT.exists():
    shutil.rmtree(OBB_PROJECT_ROOT)

shutil.copytree(DATASET_ROOT / 'images', OBB_DATASET_ROOT / 'images')
(OBB_DATASET_ROOT / 'labels').mkdir(parents=True, exist_ok=True)

converted_bbox = 0
converted_polygon = 0
kept_obb = 0
for split in ['train', 'val', 'test']:
    src_dir = DATASET_ROOT / 'labels' / split
    dst_dir = OBB_DATASET_ROOT / 'labels' / split
    dst_dir.mkdir(parents=True, exist_ok=True)
    for src_label in sorted(src_dir.glob('*.txt')):
        out_lines = []
        for line in src_label.read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            before_count = len(line.split())
            out_lines.append(label_line_to_obb_line(line))
            if before_count == 5:
                converted_bbox += 1
            elif before_count == 9:
                kept_obb += 1
            else:
                converted_polygon += 1
        (dst_dir / src_label.name).write_text('\n'.join(out_lines) + ('\n' if out_lines else ''), encoding='utf-8')

(OBB_PROJECT_ROOT / 'configs').mkdir(parents=True, exist_ok=True)
OBB_DATA_YAML.write_text(
    'path: /content/aod4_obb_work/project_obb/dataset\n'
    'train: images/train\n'
    'val: images/val\n'
    'test: images/test\n\n'
    'names:\n'
    '  0: airplane\n'
    '  1: bird\n'
    '  2: drone\n'
    '  3: helicopter\n',
    encoding='utf-8',
)

obb_sample = find_first_non_empty_label(OBB_DATASET_ROOT / 'labels' / 'train')
obb_line = obb_sample.read_text(encoding='utf-8').strip().splitlines()[0]
print('converted bbox lines:', converted_bbox)
print('converted polygon lines:', converted_polygon)
print('already OBB lines:', kept_obb)
print('OBB sample label:', obb_sample)
print('OBB sample line:', obb_line)
print('OBB field count:', len(obb_line.split()))
print('OBB data yaml:', OBB_DATA_YAML)

if len(obb_line.split()) != 9:
    raise ValueError('OBB donusumu basarisiz: label satiri 9 alanli degil.')

## Kurulum

In [ ]:
%cd /content/drive/MyDrive/AOD4/AOD4_GitHub
!python -m pip install -U pip
!python -m pip install -r requirements.txt
!python -m pip install -U ultralytics

In [ ]:
import torch
import ultralytics

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU aktif degil. Runtime > Change runtime type > GPU sec.')
print('ultralytics:', ultralytics.__version__)

## YOLO26m OBB Egitimi

Egitim eski `scripts/train_yolo.py` ile degil, yeni `AOD4_GitHub/OBB_HA_HB/scripts/train_yolo.py` ile calisir.

Script icindeki yeni parametreler korunur:

```text
model: YOLO26m-obb.pt
epochs: 100
imgsz: 640
batch: 64
name: OBB_HA_HB
patience: 15
task: obb
```

Sadece `--data` zip'ten uretilen OBB dataset config dosyasina yonlendirilir.

In [ ]:
!python OBB_HA_HB/scripts/train_yolo.py --data "/content/aod4_obb_work/project_obb/configs/data.yaml" --project "/content/drive/MyDrive/AOD4/live_runs"

## Ciktilari Kontrol Et

In [ ]:
!find "/content/drive/MyDrive/AOD4/live_runs/OBB_HA_HB" -maxdepth 3 -type f | sort | tail -n 60

In [ ]:
from pathlib import Path

best = Path('/content/drive/MyDrive/AOD4/live_runs/OBB_HA_HB/weights/best.pt')
last = Path('/content/drive/MyDrive/AOD4/live_runs/OBB_HA_HB/weights/last.pt')
print('best exists:', best.exists(), best)
print('last exists:', last.exists(), last)